# 1.3. Орбитальное движение в плоскости

Рассмотрим движение материальной точки в поле неподвижного центра притяжения:
$$\ddot{\mathbf r}=-\frac{\mathbf r}{|\mathbf r|^3}.$$
Координаты и время безразмерны. Такое уравнение получается, если единицу времени выбрать равной $\sqrt{\ell^3/(GM)}$, где $\ell$ — единица длины. Масса движущегося тела сокращается из уравнения.

Сначала проверим методы Эйлера и Рунге–Кутты четвёртого порядка на круговой орбите. Затем рассмотрим эллипсы с одинаковой большой полуосью и выясним, как эксцентриситет влияет на точность расчёта. Основные сравнения проводятся на одном обороте; анимации в конце показывают сравнение методов, ансамбль из десяти тел и закон площадей.

Ноутбук выполняется сверху вниз в Python 3. Нужны NumPy, Matplotlib и IPython. Графики и кадры анимаций сохранены в выводах; кнопка ▶ находится под каждой анимацией. При первом открытии Jupyter может потребоваться отметить файл как доверенный, чтобы разрешить воспроизведение сохранённого HTML.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Polygon
from IPython.display import HTML, Markdown, display

plt.rcParams.update({
    'figure.dpi': 100,
    'font.size': 10,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'animation.embed_limit': 100,
    'animation.frame_format': 'jpeg',  # Компактные кадры; статические графики сохраняются в PNG.
})

# Таблицы выводим без дополнительной зависимости от pandas.
def table(headers, rows):
    def line(items):
        return '| ' + ' | '.join(str(v).replace('|', '&#124;') for v in items) + ' |'
    text = [line(headers), line(['---'] * len(headers))]
    text += [line(row) for row in rows]
    display(Markdown('\n'.join(text)))

## 1. Система первого порядка и два метода

Вектор состояния содержит четыре величины: $s=(x,y,v_x,v_y)$. Тогда
$$
\dot s=F(s)=\left(v_x,\ v_y,\ -\frac{x}{r^3},\ -\frac{y}{r^3}\right),
\qquad r=\sqrt{x^2+y^2}.
$$
В явном методе Эйлера все производные вычисляются по старому состоянию:
$$s_{n+1}=s_n+hF(s_n).$$
То есть и координаты, и скорости обновляются одновременно. Для RK4 используются четыре оценки правой части:
$$
\begin{aligned}
k_1&=F(s_n), & k_2&=F(s_n+hk_1/2),\\
k_3&=F(s_n+hk_2/2), & k_4&=F(s_n+hk_3),\\
s_{n+1}&=s_n+\frac h6(k_1+2k_2+2k_3+k_4).
\end{aligned}
$$
У Эйлера глобальная ошибка на фиксированном промежутке имеет порядок $O(h)$, у RK4 — $O(h^4)$, когда шаг достаточно мал. Это проверим на известном решении.

In [ ]:
def rhs(s):
    x, y, vx, vy = s
    r = np.hypot(x, y)
    return np.array([vx, vy, -x / r**3, -y / r**3])


def euler_step(s, h):
    return s + h * rhs(s)


def rk4_step(s, h):
    k1 = rhs(s)
    k2 = rhs(s + h * k1 / 2)
    k3 = rhs(s + h * k2 / 2)
    k4 = rhs(s + h * k3)
    return s + h * (k1 + 2*k2 + 2*k3 + k4) / 6


def integrate(s0, t_end, n_steps, step):
    """n_steps одинаковых шагов; последний узел находится ровно в t_end."""
    h = t_end / n_steps
    t = np.linspace(0, t_end, n_steps + 1)
    states = np.empty((n_steps + 1, 4))
    states[0] = s0
    for i in range(n_steps):
        states[i + 1] = step(states[i], h)
    return t, states


methods = {'Эйлер': euler_step, 'RK4': rk4_step}
method_colors = {'Эйлер': 'tab:blue', 'RK4': 'tab:orange'}

## 2. Проверка на круговой орбите

Начальные условия из задания:
$$\mathbf r_0=(1,0),\qquad \mathbf v_0=(0,1).$$
Точное решение — $\mathbf r(t)=(\cos t,\sin t)$, $\mathbf v(t)=(-\sin t,\cos t)$; период $T_0=2\pi$.

Кроме положения будем следить за **удельной** полной энергией и моментом импульса:
$$
\mathcal E=\frac{v_x^2+v_y^2}{2}-\frac1r,\qquad
L_z=xv_y-yv_x.
$$
Их точные значения для этой окружности равны $-1/2$ и $1$. Постоянство энергии следует из $\mathbf v\cdot\dot{\mathbf v}+\dot r/r^2=0$, а момента — из того, что ускорение направлено вдоль радиуса. Поэтому дрейф этих величин в расчёте характеризует численную ошибку.

Возьмём $N=250,500,1000,2000,4000$ шагов за оборот, то есть $h=T_0/N$. Ошибку положения измерим во всех узлах:
$$\varepsilon(h)=\max_n\|\mathbf r_h(t_n)-(\cos t_n,\sin t_n)\|.$$
В таблице также приведём максимальные относительные отклонения энергии и момента от начальных значений и наблюдаемый порядок
$$p=\log_2\frac{\varepsilon(h)}{\varepsilon(h/2)}.$$
Для первой сетки порядок ещё не с чем сравнить.

In [ ]:
def energy_and_momentum(states):
    radius = np.linalg.norm(states[:, :2], axis=1)
    speed2 = np.sum(states[:, 2:]**2, axis=1)
    energy = speed2 / 2 - 1 / radius
    momentum = states[:, 0]*states[:, 3] - states[:, 1]*states[:, 2]
    return energy, momentum


T0 = 2 * np.pi
circle_initial = np.array([1., 0., 0., 1.])
Ns = np.array([250, 500, 1000, 2000, 4000])
circle_runs = {}
circle_errors = {}
rows = []

for name, step in methods.items():
    previous_error = None
    circle_errors[name] = []
    for N in Ns:
        t, states = integrate(circle_initial, T0, N, step)
        reference = np.column_stack((np.cos(t), np.sin(t)))
        error = np.max(np.linalg.norm(states[:, :2] - reference, axis=1))
        energy, momentum = energy_and_momentum(states)
        energy_drift = np.max(np.abs(energy + 0.5)) / 0.5
        momentum_drift = np.max(np.abs(momentum - 1))
        order = '—' if previous_error is None else f'{np.log2(previous_error/error):.2f}'
        rows.append([name, N, f'{error:.3e}', order,
                     f'{energy_drift:.3e}', f'{momentum_drift:.3e}'])
        circle_runs[name, N] = (t, states)
        circle_errors[name].append(error)
        previous_error = error

table(['Метод', 'N', 'Макс. ошибка положения', 'p',
       'Макс. отн. дрейф энергии', 'Макс. отн. дрейф Lz'], rows)

На последних двух сетках получаются наблюдаемые порядки, близкие к 1 и 4. Это показывает, что реализованные схемы дают ожидаемую сходимость.

На первом рисунке оставим $N=500$, чтобы было видно искажение орбиты Эйлером. На втором покажем зависимость ошибки от шага для всех расчётов из таблицы.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.2))
angle = np.linspace(0, 2*np.pi, 500)
axes[0].plot(np.cos(angle), np.sin(angle), 'k--', lw=1.6, label='Точное решение')
for name in methods:
    t, states = circle_runs[name, 500]
    energy, momentum = energy_and_momentum(states)
    color = method_colors[name]
    axes[0].plot(states[:, 0], states[:, 1], color=color, label=name)
    axes[0].plot(states[-1, 0], states[-1, 1], 'o', color=color, ms=4)
    axes[1].plot(t/T0, energy, color=color, label=name)
    axes[2].plot(t/T0, momentum, color=color, label=name)
axes[0].plot(0, 0, 'k*', ms=9)
axes[0].set(xlabel='x', ylabel='y', aspect='equal', title='Орбита, h = T₀/500')
axes[1].axhline(-0.5, color='k', ls='--', lw=1, label='Точно')
axes[1].set(xlabel='t/T₀', ylabel='Энергия', title='Полная энергия')
axes[2].axhline(1, color='k', ls='--', lw=1, label='Точно')
axes[2].set(xlabel='t/T₀', ylabel='Lz', title='Момент импульса')
axes[0].legend(fontsize=8, loc='upper center', bbox_to_anchor=(.5, -.16), ncol=3)
for ax in axes[1:]:
    ax.legend(fontsize=8, loc='best')
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
steps = T0 / Ns
for ax, name, p in zip(axes, methods, [1, 4]):
    errors = np.array(circle_errors[name])
    ax.loglog(steps, errors, 'o-', label='Расчёт')
    ax.loglog(steps, errors[-1]*(steps/steps[-1])**p, 'k--', lw=1,
              label=rf'$\propto h^{p}$')
    ax.set(xlabel='Шаг h', ylabel='Максимальная ошибка положения', title=name)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

У Эйлера орбита уходит наружу, энергия растёт, а тело через $T_0$ заметно отстаёт от начальной точки. У RK4 на таком масштабе отклонения почти не видны; их величину даёт таблица. При $N=4000$ максимальная ошибка положения за оборот составляет около $0.094$ у Эйлера и $9\cdot10^{-13}$ у RK4.

Рост момента импульса у явного Эйлера можно объяснить непосредственно из схемы. Подставив $\mathbf r_{n+1}=\mathbf r_n+h\mathbf v_n$ и $\mathbf v_{n+1}=\mathbf v_n-h\mathbf r_n/r_n^3$ в векторное произведение, получаем
$$L_{z,n+1}=L_{z,n}\left(1+\frac{h^2}{r_n^3}\right).$$
При положительном $L_z$ каждое обновление увеличивает его. Уменьшение шага уменьшает этот эффект, но при конечном шаге точного сохранения нет.

## 3. Как задать эллипсы с одной большой полуосью

Пусть большая полуось $a=1$, а эксцентриситет принимает значения $e=0,\ 0.3,\ 0.6,\ 0.8$. Центр притяжения находится в **фокусе** эллипса. Начальная точка — перицентр на положительной оси $x$:
$$r_p=a(1-e),\qquad r_a=a(1+e),\qquad b=a\sqrt{1-e^2}.$$
Здесь $r_a$ — расстояние до апоцентра, $b$ — малая полуось. Центр самого эллипса находится в точке $(-ae,0)$.

Для эллиптической орбиты $\mathcal E=-1/(2a)$. В перицентре скорость перпендикулярна радиусу, поэтому из закона энергии
$$
\frac{v_p^2}{2}-\frac{1}{a(1-e)}=-\frac{1}{2a}
\quad\Longrightarrow\quad
v_p=\sqrt{\frac{1+e}{a(1-e)}}.
$$
Значит, нужные начальные условия имеют вид
$$s_0=\left(a(1-e),\ 0,\ 0,\ \sqrt{\frac{1+e}{a(1-e)}}\right).$$
При таком выборе энергия у всех тел одинакова, а аналитический период равен $T=2\pi a^{3/2}$ и от $e$ не зависит. Момент импульса при этом различается: $L_z=\sqrt{a(1-e^2)}$. Независимость периода от $e$ можно также получить из закона площадей: площадь эллипса $\pi ab$ делим на постоянную секторную скорость $L_z/2$,
$$T=\frac{2\pi ab}{L_z}=2\pi a^{3/2}.$$

In [ ]:
a = 1.0
eccentricities = [0.0, 0.3, 0.6, 0.8]
T = 2 * np.pi * a**1.5


def initial_state(e, a=1.0):
    return np.array([a*(1-e), 0., 0., np.sqrt((1+e)/(a*(1-e)))])


initial_states = np.array([initial_state(e, a) for e in eccentricities])
initial_energy, initial_momentum = energy_and_momentum(initial_states)
table(['e', 'Перицентр rp', 'Апоцентр ra', 'Малая полуось b', 'Начальная скорость vp', 'Энергия'],
      [[f'{e:.1f}', f'{a*(1-e):.3f}', f'{a*(1+e):.3f}',
        f'{a*np.sqrt(1-e**2):.3f}', f'{s0[3]:.3f}', f'{en:.3f}']
       for e, s0, en in zip(eccentricities, initial_states, initial_energy)])
np.testing.assert_allclose(initial_energy, -1/(2*a), atol=1e-14)
np.testing.assert_allclose(initial_momentum, np.sqrt(a*(1-np.array(eccentricities)**2)))

### Аналитическая форма и положение во времени

Для построения эллипса достаточно параметризации
$$x=a(\cos u-e),\qquad y=b\sin u,\qquad 0\leq u\leq2\pi.$$
Параметр $u$ называется эксцентрической аномалией. Он не равен полярному углу и при $e>0$ не растёт равномерно со временем. Поэтому для точной отметки движущегося тела нужно дополнительно решить уравнение Кеплера:
$$u-e\sin u=nt,\qquad n=a^{-3/2}.$$
В функции `kepler_position` оно решается методом Ньютона. Производная левой части $1-e\cos u$ положительна для $e<1$; в рассматриваемом диапазоне до $e=0.8$ начальное приближение $u=nt$ даёт сходимость. Эта функция служит независимым эталоном для положения; уравнение движения по-прежнему интегрируется методами выше.

In [ ]:
def ellipse_curve(e, a=1.0, n_points=600):
    u = np.linspace(0, 2*np.pi, n_points)
    return np.column_stack((a*(np.cos(u)-e), a*np.sqrt(1-e**2)*np.sin(u)))


def kepler_position(t, e, a=1.0):
    mean_anomaly = np.atleast_1d(t) / a**1.5
    u = mean_anomaly.copy()
    for _ in range(40):
        correction = (u - e*np.sin(u) - mean_anomaly) / (1 - e*np.cos(u))
        u -= correction
        if np.max(np.abs(correction)) < 1e-13:
            break
    else:
        raise RuntimeError('Уравнение Кеплера не сошлось')
    return np.column_stack((a*(np.cos(u)-e), a*np.sqrt(1-e**2)*np.sin(u)))


# При e=0 эллиптический эталон должен переходить в круговой.
t_check = np.linspace(0, T0, 20)
np.testing.assert_allclose(kepler_position(t_check, 0),
                           np.column_stack((np.cos(t_check), np.sin(t_check))), atol=1e-13)

### Измерение периода по численной траектории

За период примем время следующего пересечения положительной полуоси $x$ в направлении снизу вверх: $y$ меняет знак с минуса на плюс при $x>0$. У точной орбиты это возвращение в перицентр; для окружности тот же способ определяет полный оборот. Начальное пересечение при $t=0$ не учитывается.

Если брать только номер ближайшего узла, ошибка измерения времени будет порядка $h$ и скроет точность RK4. Поэтому после обнаружения пересечения уточним его внутри последнего шага: из состояния перед пересечением делаем один шаг RK4 длины $\tau\in[0,h]$ и бисекцией находим, при каком $\tau$ получится $y=0$. Это уточняет место события, но не устраняет ошибку, накопленную до него.

Форму орбиты дополнительно проверим через невязку уравнения эллипса:
$$q(x,y)=\frac{(x+ae)^2}{a^2}+\frac{y^2}{a^2(1-e^2)}-1.$$
Это безразмерная невязка, а не расстояние до эллипса. Для любой точки точной орбиты она равна нулю. Так можно заметить отклонение формы, даже если на рисунке линии сливаются.

In [ ]:
def orbital_period(t, states):
    crossings = np.flatnonzero(
        (states[:-1, 1] < 0) & (states[1:, 1] >= 0) & (states[1:, 0] > 0)
    )
    if len(crossings) == 0:
        raise RuntimeError('До конца расчёта полный оборот не завершён')
    i = crossings[0]
    left, right = 0.0, t[i+1] - t[i]
    for _ in range(40):
        middle = (left + right) / 2
        if rk4_step(states[i], middle)[1] < 0:
            left = middle
        else:
            right = middle
    return t[i] + (left + right) / 2


# Запас по времени нужен только для регистрации пересечения.
# При N шагах за период здесь получится 5N/4 шагов до 1.25T.
ellipse_runs = {}
period_values = {}
shape_residuals = {}
for e in eccentricities:
    for N in [2000, 4000]:
        t, states = integrate(initial_state(e, a), 1.25*T, 5*N//4, rk4_step)
        period_values[e, N] = orbital_period(t, states)
        if N == 4000:
            ellipse_runs[e] = (t[:N+1], states[:N+1])
            x, y = states[:N+1, 0], states[:N+1, 1]
            q = ((x+a*e)/a)**2 + y**2/(a*a*(1-e*e)) - 1
            shape_residuals[e] = np.max(np.abs(q))

table(['e', 'Период, N=4000', 'Отн. ошибка T, N=2000', 'Отн. ошибка T, N=4000', 'max abs(q), N=4000'],
      [[f'{e:.1f}', f'{period_values[e, 4000]:.10f}', f'{abs(period_values[e, 2000]/T-1):.3e}',
        f'{abs(period_values[e, 4000]/T-1):.3e}', f'{shape_residuals[e]:.3e}']
       for e in eccentricities])
print(f'Аналитический период для всех орбит: T = {T:.10f}')

Ниже чёрная кривая — аналитический эллипс. Синие точки взяты из численного решения через одинаковые промежутки $T/40$. Их сгущение около апоцентра показывает, что там тело движется медленнее. Звёздочка отмечает притягивающий центр, серый крестик — геометрический центр эллипса.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.5, 6.4), sharex=True, sharey=True)
for ax, e in zip(axes.flat, eccentricities):
    curve = ellipse_curve(e, a)
    t, states = ellipse_runs[e]
    ax.plot(curve[:, 0], curve[:, 1], 'k-', lw=1.2, label='Точный эллипс')
    ax.plot(states[::100, 0], states[::100, 1], 'o', ms=3.5, color='tab:blue',
            label='RK4, точки через T/40')
    ax.plot(0, 0, 'k*', ms=9)
    ax.plot(-a*e, 0, '+', color='0.5', ms=7)
    ax.set(xlim=(-1.95*a, 1.15*a), ylim=(-1.15*a, 1.15*a), aspect='equal',
           xlabel='x', ylabel='y', title=f'e = {e:g}')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()

Численные периоды близки к одному и тому же $2\pi$. Сильнее всего отличается результат для $e=0.8$; при уменьшении шага вдвое отклонение уменьшается. На мелкой сетке относительная ошибка периода для этой орбиты составляет около $10^{-8}$. Следовательно, различие периодов в таблице объясняется ошибкой интегрирования, а не физической зависимостью периода от эксцентриситета.

При увеличении $e$ перицентр приближается к фокусу, апоцентр удаляется, малая полуось уменьшается. Расстояние между перицентром и апоцентром остаётся равным $2a$. На рисунках и по невязке $q$ численные траектории согласуются с этой геометрией.

## 4. Возвращение в начальную точку и уменьшение шага

Теперь будем сравнивать положения **строго в аналитический момент** $T$, а не в измеренный численный период:
$$\varepsilon_r(h)=\frac{\|\mathbf r_h(T)-\mathbf r_0\|}{a}.$$
Деление на $a$ делает ошибку относительной к размеру орбиты. При $a=1$ её численное значение совпадает с абсолютной ошибкой. В каждом расчёте берём ровно $N$ шагов $h=T/N$, поэтому конечную точку не приходится интерполировать.

Повторим расчёт обоими методами. Для каждого фиксированного $e$ начальные условия при изменении шага остаются прежними.

In [ ]:
return_errors = {}
for name, step in methods.items():
    for e in eccentricities:
        errors = []
        s0 = initial_state(e, a)
        for N in Ns:
            _, states = integrate(s0, T, N, step)
            errors.append(np.linalg.norm(states[-1, :2] - s0[:2]) / a)
        return_errors[name, e] = np.array(errors)

for name in methods:
    display(Markdown(f'**{name}: ошибка возвращения**'))
    table(['e'] + [f'N={N}' for N in Ns],
          [[f'{e:.1f}'] + [f'{v:.3e}' for v in return_errors[name, e]]
           for e in eccentricities])

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
for ax, name in zip(axes, methods):
    for e in eccentricities:
        ax.loglog(Ns, return_errors[name, e], 'o-', label=f'e={e:g}')
    ax.set(xlabel='Число шагов N за период', ylabel='Ошибка возвращения / a', title=name)
    ax.set_xticks(Ns, [str(N) for N in Ns])
    ax.xaxis.set_minor_locator(plt.NullLocator())
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

При одинаковом числе шагов вытянутая орбита считается заметно хуже. Например, для RK4 при $N=250$ ошибка возрастает от $6.6\cdot10^{-8}$ для круга до примерно $0.026$ при $e=0.8$. При $N=4000$ ошибка для $e=0.8$ уже около $2\cdot10^{-7}$. На последних двух сетках отношения ошибок RK4 близки к 16, что соответствует четвёртому порядку.

У Эйлера даже $4000$ шагов недостаточно для приемлемого возвращения при больших $e$. При $e=0.8$ ошибка сравнима с размером орбиты. Здесь ещё нельзя ожидать, что каждое уменьшение шага вдвое точно уменьшит ошибку вдвое: рассчитанная траектория слишком сильно отличается от точной.

Для определённости зададим требование $\varepsilon_r<10^{-6}$ и найдём первое подходящее $N$ **среди проверенных сеток**. Это выбранный нами допуск на положение через один оборот; условие задачи фиксированного допуска не задаёт.

In [ ]:
return_tolerance = 1e-6
required_N = {}
rows = []
for e in eccentricities:
    errors = return_errors['RK4', e]
    suitable = np.flatnonzero(errors < return_tolerance)
    first = int(suitable[0]) if len(suitable) else None
    required_N[e] = None if first is None else int(Ns[first])
    last_order = np.log2(errors[-2]/errors[-1])
    rows.append([f'{e:.1f}', required_N[e] if first is not None else 'не достигнуто',
                 '—' if first is None else f'{errors[first]:.3e}', f'{last_order:.2f}'])
table(['e', 'Первое N с ошибкой < 10⁻⁶', 'Ошибка на этой сетке', 'p на двух последних сетках'], rows)

## 5. Скорость вдоль орбиты и требование к шагу

Из закона энергии следует формула $v^2=2/r-1/a$. Поэтому скорость максимальна в перицентре и минимальна в апоцентре:
$$
v_p=\sqrt{\frac{1+e}{a(1-e)}},\qquad
v_a=\sqrt{\frac{1-e}{a(1+e)}},\qquad
\frac{v_p}{v_a}=\frac{1+e}{1-e}.
$$
Для $e=0.8$ получаются $v_p=3$, $v_a=1/3$ и отношение скоростей 9. При этом большая скорость приходится на участок с наименьшим расстоянием до центра, где ускорение $1/r^2$ также максимально.

Оценим местный масштаб времени как
$$\tau_p=\frac{r_p}{v_p}=a^{3/2}\frac{(1-e)^{3/2}}{\sqrt{1+e}}.$$
За такое время вблизи перицентра радиус-вектор поворачивается на угол порядка одного радиана. Для разрешения этого движения нужно $h\ll\tau_p$. Это оценка масштаба, а фактическую точность всё равно проверяем уменьшением шага, как в предыдущем разделе.

In [ ]:
rows = []
fig, ax = plt.subplots(figsize=(8, 3.8))
for e in eccentricities:
    t, states = ellipse_runs[e]
    speed = np.linalg.norm(states[:, 2:], axis=1)
    ax.plot(t/T, speed, label=f'e={e:g}')
    vp = np.sqrt((1+e)/(a*(1-e)))
    va = np.sqrt((1-e)/(a*(1+e)))
    tau_p = a*(1-e)/vp
    rows.append([f'{e:.1f}', f'{vp:.3f}', f'{va:.3f}', f'{vp/va:.2f}',
                 f'{tau_p:.4f}', f'{(T/250)/tau_p:.3f}'])
ax.set(xlabel='t/T', ylabel='Скорость', xlim=(0, 1), title='Скорость за один оборот, RK4')
ax.legend(fontsize=9, loc='upper center', ncol=4)
fig.tight_layout()
plt.show()
table(['e', 'vp', 'va', 'vp/va', 'τp', 'h/τp при N=250'], rows)

Для круга $\tau_p=1$, а для $e=0.8$ — $1/15$. При $N=250$ на этот короткий масштаб у перицентра приходится всего около 2.7 шага. Поэтому равномерная сетка, вполне точная для круга при RK4, оказывается грубой для вытянутого эллипса. Это объясняет рост ошибки возвращения с эксцентриситетом.

Период при этом остаётся прежним: быстрый проход перицентра компенсируется медленным движением вдали от центра. На графике минимум скорости приходится на $t=T/2$, когда тело достигает апоцентра.

## 6. Анимации

Во всех анимациях положение берётся из численного решения. Шаг интегрирования и интервал между кадрами различаются: для плавного рисунка показываем только часть рассчитанных узлов. В каждой сцене есть один общий счётчик времени. Функция ниже только превращает кадры Matplotlib в встроенный HTML; численные данные она не меняет.

In [ ]:
def show_animation(animation):
    html = animation.to_jshtml(default_mode='once')
    # Текстовые значки работают и без загрузки внешнего шрифта.
    html = re.sub(r'<link\s+rel="stylesheet"\s+href="[^"]+">', '', html)
    icons = {'fa-play': '▶', 'fa-play fa-flip-horizontal': '◀', 'fa-pause': 'Ⅱ',
             'fa-step-forward': '|▶', 'fa-step-backward': '◀|',
             'fa-fast-forward': '⏭', 'fa-fast-backward': '⏮',
             'fa-minus': '−', 'fa-plus': '+'}
    html = re.sub(r'<i class="fa ([^"]+)"[^>]*></i>', lambda m: icons.get(m[1], m[1]), html)
    display(HTML(html))

### Эйлер и RK4: одинаковый шаг, одинаковое время

Выберем $e=0.6$ и $h=T/1000$. Для RK4 этот шаг уже даёт ошибку возвращения меньше $10^{-6}$, а у Эйлера расхождение хорошо видно. На обеих панелях одинаковый масштаб. Пунктир показывает точную форму, красный пустой кружок — точное положение в текущий момент, цветная точка и след — численное решение. Подпись $\varepsilon_r(t)$ показывает расстояние от численного положения до точного, делённое на $a$.

In [ ]:
def animate_methods(e=0.6, N=1000):
    times = np.linspace(0, T, N+1)
    paths = {name: integrate(initial_state(e, a), T, N, step)[1]
             for name, step in methods.items()}
    frame_indices = np.arange(0, N+1, 10)
    exact = kepler_position(times, e, a)
    curve = ellipse_curve(e, a)
    all_xy = np.vstack([curve] + [path[:, :2] for path in paths.values()])
    low, high = all_xy.min(axis=0) - .15*a, all_xy.max(axis=0) + .15*a

    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.5), dpi=85, sharex=True, sharey=True)
    artists = []
    for ax, name, color in zip(axes, methods, ['tab:blue', 'tab:orange']):
        ax.plot(curve[:, 0], curve[:, 1], 'k--', lw=1, label='Точная орбита')
        ax.plot(0, 0, 'k*', ms=9)
        trail, = ax.plot([], [], color=color, lw=1.5)
        body, = ax.plot([], [], 'o', color=color, ms=5, label=name)
        reference, = ax.plot([], [], 'o', mfc='none', mec='tab:red', ms=8,
                              label='Точное положение')
        error_text = ax.text(.03, .96, '', transform=ax.transAxes, va='top', fontsize=9)
        ax.set(xlim=(low[0], high[0]), ylim=(low[1], high[1]), aspect='equal',
               xlabel='x', ylabel='y', title=name)
        ax.legend(loc='lower left', fontsize=7)
        artists.append((trail, body, reference, error_text))
    title = fig.suptitle('')
    fig.subplots_adjust(left=.08, right=.99, bottom=.15, top=.83, wspace=.15)

    def update(frame):
        i = frame_indices[frame]
        for (name, path), (trail, body, reference, text) in zip(paths.items(), artists):
            trail.set_data(path[:i+1, 0], path[:i+1, 1])
            body.set_data([path[i, 0]], [path[i, 1]])
            reference.set_data([exact[i, 0]], [exact[i, 1]])
            error = np.linalg.norm(path[i, :2] - exact[i]) / a
            text.set_text(f'εr(t) = {error:.2e}')
        title.set_text(f'e = {e:g},  h = T/{N},  t/T = {times[i]/T:.2f}')

    animation = FuncAnimation(fig, update, frames=len(frame_indices), interval=65, blit=False)
    plt.close(fig)
    return animation


methods_animation = animate_methods()
show_animation(methods_animation)

### Десять тел с разными эксцентриситетами

Пусть $e$ равномерно меняется от 0 до 0.8. Все тела запускаются одновременно из своих перицентров, все имеют $a=1$ и взаимодействуют только с неподвижным центром. Для всех взят один шаг RK4 $T/4000$, выбранный по самой вытянутой из рассмотренных орбит. Точки показываются через $T/200$.

После старта тела проходят разные углы относительно центра; к $T/2$ одновременно оказываются в своих апоцентрах, а к $T$ возвращаются к перицентрам. Наибольшая ошибка возвращения для всего ансамбля напечатана под анимацией.

In [ ]:
def animate_ensemble():
    es = np.linspace(0, .8, 10)
    N = 4000
    frame_indices = np.arange(0, N+1, 20)
    paths = [integrate(initial_state(e, a), T, N, rk4_step)[1] for e in es]
    colors = plt.cm.viridis(np.linspace(.05, .9, len(es)))
    fig, ax = plt.subplots(figsize=(7.6, 4.8), dpi=85)
    dots = []
    for e, color in zip(es, colors):
        curve = ellipse_curve(e, a)
        ax.plot(curve[:, 0], curve[:, 1], color=color, lw=.7, alpha=.65)
        dot, = ax.plot([], [], 'o', ms=5, color=color, label=f'{e:.2f}')
        dots.append(dot)
    ax.plot(0, 0, 'k*', ms=10)
    ax.set(xlim=(-1.95*a, 1.15*a), ylim=(-1.15*a, 1.15*a),
           aspect='equal', xlabel='x', ylabel='y')
    ax.legend(loc='center left', bbox_to_anchor=(1.01, .5), title='e', fontsize=8)
    title = ax.set_title(f'a = {a:g},  общее время t/T = 0.00')
    fig.tight_layout()

    def update(frame):
        i = frame_indices[frame]
        for path, dot in zip(paths, dots):
            dot.set_data([path[i, 0]], [path[i, 1]])
        title.set_text(f'a = {a:g},  общее время t/T = {i/N:.2f}')

    animation = FuncAnimation(fig, update, frames=len(frame_indices), interval=40, blit=False)
    plt.close(fig)
    errors = [np.linalg.norm(path[-1, :2] - path[0, :2])/a for path in paths]
    return animation, max(errors)


ensemble_animation, ensemble_return_error = animate_ensemble()
show_animation(ensemble_animation)
print(f'Максимальная ошибка возвращения / a для десяти тел: {ensemble_return_error:.3e}')

### Равные площади за равные времена

Возьмём $e=0.6$ и разделим период на 12 равных интервалов. Площадь, заметённая радиусом-вектором, удовлетворяет
$$\frac{dS}{dt}=\frac12(xv_y-yv_x)=\frac{L_z}{2}.$$
Поскольку $L_z$ постоянен, за одинаковые $\Delta t=T/12$ заметутся одинаковые площади
$$S_* = \frac{\pi ab}{12}.$$
Близко к перицентру такой сектор имеет большой угол и короткие радиусы, а около апоцентра — меньший угол и длинные радиусы.

Для проверки посчитаем площади по численной траектории RK4. Каждая область ограничена двумя радиусами и дугой орбиты. Заменив дугу последовательностью отрезков, получим сумму площадей треугольников:
$$S_i\approx\frac12\sum_j(x_jy_{j+1}-x_{j+1}y_j).$$
Знак здесь положителен, так как движение идёт против часовой стрелки. На границах секторов есть узлы сетки: на каждый интервал приходится 400 шагов. Результат сравним с $S_*$, а не только с площадями соседних секторов.

In [ ]:
e_area = .6
n_sectors = 12
steps_per_sector = 400
N_area = n_sectors * steps_per_sector
area_t, area_states = integrate(initial_state(e_area, a), T, N_area, rk4_step)
area_xy = area_states[:, :2]
area_speed = np.linalg.norm(area_states[:, 2:], axis=1)
sector_areas = []
for j in range(n_sectors):
    segment = area_xy[j*steps_per_sector:(j+1)*steps_per_sector+1]
    cross_products = segment[:-1, 0]*segment[1:, 1] - segment[:-1, 1]*segment[1:, 0]
    sector_areas.append(np.sum(cross_products)/2)
sector_areas = np.array(sector_areas)
sector_exact = np.pi*a*a*np.sqrt(1-e_area**2)/n_sectors
print(f'Теоретическая площадь сектора: {sector_exact:.8f}')
print(f'Численные площади: от {sector_areas.min():.8f} до {sector_areas.max():.8f}')
print(f'Наибольшее относительное отклонение от теории: {np.max(np.abs(sector_areas/sector_exact-1)):.3e}')

Небольшая разница площадей ожидаема: к ошибке интегрирования добавляется замена гладкой дуги хордой. При выбранной сетке она составляет лишь несколько миллионных долей площади сектора. На рисунке ниже можно рассмотреть все области сразу; следующая анимация показывает их последовательное заполнение. Цвета временных интервалов на графике скорости соответствуют цветам секторов.

In [ ]:
def animate_sectors():
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.3), dpi=85)
    colors = plt.cm.tab20(np.linspace(0, .95, n_sectors))
    axes[0].plot(area_xy[:, 0], area_xy[:, 1], 'k-', lw=1)
    axes[0].plot(0, 0, 'k*', ms=9, zorder=5)
    patches = []
    for j, color in enumerate(colors):
        patch = Polygon(np.zeros((3, 2)), facecolor=color, alpha=.65, edgecolor='none')
        axes[0].add_patch(patch)
        patches.append(patch)
        axes[1].axvspan(j/n_sectors, (j+1)/n_sectors, color=color, alpha=.25)
    radius, = axes[0].plot([], [], 'k-', lw=1)
    body, = axes[0].plot([], [], 'ko', ms=5)
    axes[0].set(xlim=(-1.8*a, .6*a), ylim=(-a, a), aspect='equal', xlabel='x', ylabel='y')
    axes[1].plot(area_t/T, area_speed, color='0.15', lw=1.5)
    cursor = axes[1].axvline(0, color='tab:red', lw=1.5)
    speed_point, = axes[1].plot([], [], 'o', color='tab:red', ms=5)
    axes[1].set(xlim=(-.02, 1.02), xlabel='t/T', ylabel='Скорость')
    title = fig.suptitle('')
    fig.tight_layout(rect=(0, 0, 1, .94))
    frame_indices = np.arange(0, N_area+1, 25)  # 193 кадра; границы секторов входят в кадры

    def update(frame):
        i = frame_indices[frame]
        for j, patch in enumerate(patches):
            start = j*steps_per_sector
            end = min(i, (j+1)*steps_per_sector)
            patch.set_visible(end > start)
            if end > start:
                patch.set_xy(np.vstack(([0., 0.], area_xy[start:end+1], [0., 0.])))
        radius.set_data([0, area_xy[i, 0]], [0, area_xy[i, 1]])
        body.set_data([area_xy[i, 0]], [area_xy[i, 1]])
        cursor.set_xdata([area_t[i]/T, area_t[i]/T])
        speed_point.set_data([area_t[i]/T], [area_speed[i]])
        title.set_text(f'e = {e_area:g},  t/T = {area_t[i]/T:.2f},  Δt = T/12')

    update(len(frame_indices)-1)
    display(fig)
    animation = FuncAnimation(fig, update, frames=len(frame_indices), interval=40, blit=False)
    plt.close(fig)
    return animation


sectors_animation = animate_sectors()
show_animation(sectors_animation)

## 7. Выводы

Численный тест на окружности подтвердил первый порядок Эйлера и четвёртый порядок RK4. На одинаковой сетке явный Эйлер существенно меняет энергию и момент импульса; в рассмотренных расчётах RK4 гораздо точнее сохраняет форму и положение тела.

При общей большой полуоси $a=1$ периоды всех эллипсов равны $2\pi$. Измерение по численной траектории и повторение с меньшим шагом согласуются с этим результатом. Эксцентриситет меняет распределение скорости: при $e=0.8$ тело проходит перицентр со скоростью 3, а апоцентр — со скоростью $1/3$. Равенство заметённых площадей за равные времена подтверждено непосредственно по численной траектории.

Чем больше эксцентриситет, тем меньший постоянный шаг нужен для точного прохождения перицентра. Для выбранного допуска на возвращение $\varepsilon_r<10^{-6}$ среди проверенных сеток RK4 хватило $N=250$ при $e=0$ и $0.3$, $N=1000$ при $e=0.6$ и $N=4000$ при $e=0.8$. Эти числа относятся к положению после одного оборота и выбранным начальным условиям.

---

Условие: [README курса, задача 1.3](https://github.com/yartmihant/Students6Year2026/blob/fd8e1dc40e406aaf62d9276f889f93e5c7dbdccd/README.md). Материалы для сверки схем и оформления: [код первого занятия](https://github.com/yartmihant/Students6Year2026/tree/fd8e1dc40e406aaf62d9276f889f93e5c7dbdccd/lesson1), [примеры домашних работ](https://github.com/yartmihant/Students6Year2026/tree/fd8e1dc40e406aaf62d9276f889f93e5c7dbdccd/homework).